<a href="https://colab.research.google.com/github/ahamonk/Enterprise-Knowledge-Silos/blob/main/Enterprise_Knowledge_Silos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Enterprise Knowledge Silos

Enterprise Knowledge Assistant using:
- LangChain 1.3.9
- LangChain Community 0.4.2
- LangChain Groq 1.1.3
- FAISS Vector Database
- HuggingFace Embeddings
- Groq Llama 3.3 70B

Architecture:

Documents → Chunking → Embeddings → Vector DB → Query → Retrieval → LLM Response


In [ ]:
!pip install -q langchain langchain-community langchain-groq langchain-text-splitters faiss-cpu sentence-transformers pypdf beautifulsoup4 requests

In [1]:
!pip install python-dotenv

In [2]:
from getpass import getpass

# Prompt for the API key without displaying it
groq_api_key = getpass("Enter your Groq API Key: ")

# Save it to a .env file
with open(".env", "w") as f:
    f.write(f"GROQ_API_KEY={groq_api_key}\n")

print("✅ .env file created successfully.")

Enter your Groq API Key: ··········
✅ .env file created successfully.


In [3]:
from dotenv import load_dotenv
import os

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

print("API key loaded successfully!")

API key loaded successfully!


In [ ]:
import os
print(os.listdir())

In [ ]:
import os
import json
import requests

from bs4 import BeautifulSoup

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq



In [ ]:
def load_markdown(path):
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    return [Document(
        page_content=text,
        metadata={"source": path, "type": "README"}
    )]

def load_pdf(path):

    docs = PyPDFLoader(path).load()

    if "api" in path.lower():

        source_type = "API_DOC"

    elif "docs" in path.lower():

        source_type = "DOCUMENTATION"

    elif "rag" in path.lower():

        source_type = "WHITEPAPER"

    else:

        source_type = "PDF"

    for doc in docs:

        doc.metadata["type"] = source_type

    return docs

def load_webpage(url, doc_type):
    html = requests.get(url).text

    soup = BeautifulSoup(html, "html.parser")

    text = soup.get_text(
        separator=" ",
        strip=True
    )

    return [Document(
        page_content=text,
        metadata={
            "source": url,
            "type": doc_type
        }
    )]

def load_product_spec(path):
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    return [Document(
        page_content=text,
        metadata={
            "source": path,
            "type": "PRODUCT_SPEC"
        }
    )]

def load_slack_export(path):
    with open(path, "r") as f:
        data = json.load(f)

    docs = []

    for msg in data:
        if "text" in msg:
            docs.append(
                Document(
                    page_content=msg["text"],
                    metadata={
                        "source": path,
                        "type": "SLACK"
                    }
                )
            )

    return docs

def load_email_archive(path):
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    return [Document(
        page_content=text,
        metadata={
            "source": path,
            "type": "EMAIL"
        }
    )]

In [ ]:
documents = []

documents.extend(load_markdown("README.md"))
documents.extend(load_pdf("rag_paper.pdf"))
documents.extend(load_pdf("documentation.pdf"))
documents.extend(load_pdf("API.pdf"))

print("Documents Loaded:", len(documents))

In [ ]:
from collections import Counter

print(
    Counter(
        doc.metadata["type"]
        for doc in documents
    )
)

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings Model Loaded")

In [ ]:
vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

print("Vector DB Created")

In [ ]:
vectorstore.save_local(
    "enterprise_vector_db"
)

print("Vector DB Saved")

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

print("Retriever Created")

In [ ]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

print("Groq Model Loaded")

In [ ]:
prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are an Enterprise Knowledge Assistant.

Answer ONLY using the provided context.

If the answer is not available in the context, say that the information is not available.

Context:
{context}

Question:
{question}

Answer:"""
)

chain = prompt_template | llm
print("Prompt Template and Chain Created")


In [ ]:
def ask(question):

    retrieved_docs = retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content
        for doc in retrieved_docs
    )

    response = chain.invoke(
        {
            "context": context,
            "question": question
        }
    )

    return response.content, retrieved_docs



In [ ]:
ask("What is Retrieval Augmented Generation?")

In [ ]:
ask("How do retrievers work in LangChain?")

In [ ]:
ask("What are vector stores?")

In [ ]:
# Install Gradio
!pip -q install gradio

In [ ]:
import gradio as gr

# Function that Gradio will call
def chatbot(question):

    if not question.strip():
        return "Please enter a question.", ""

    answer, docs = ask(question)

    sources = ""

    for i, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source", "Unknown")
        page = doc.metadata.get("page", "N/A")

        sources += f"### Source {i}\n"
        sources += f"**Document:** {source}\n\n"
        sources += f"**Page:** {page}\n\n"
        sources += doc.page_content[:500] + "...\n\n---\n\n"

    return answer, sources


with gr.Blocks(title="Enterprise Knowledge Assistant") as demo:

    gr.Markdown(
        """
        # 📚 Enterprise Knowledge Assistant

        Ask questions about the enterprise knowledge base.

        **Powered by**
        - LangChain
        - FAISS
        - HuggingFace Embeddings
        - Groq Llama 3.3
        """
    )

    question = gr.Textbox(
        label="Ask a Question",
        placeholder="Example: What is Retrieval Augmented Generation?"
    )

    ask_button = gr.Button("Ask")

    answer = gr.Markdown(label="Answer")

    with gr.Accordion("Retrieved Sources", open=False):
        sources = gr.Markdown()

    ask_button.click(
        fn=chatbot,
        inputs=question,
        outputs=[answer, sources]
    )

    question.submit(
        fn=chatbot,
        inputs=question,
        outputs=[answer, sources]
    )

demo.launch(debug=True)